# 02 - Data Preprocessing

**Goal:** turn the raw IoT-23 connection logs into clean numbers that a machine learning model can use, then split them into a *training* set and a *test* set.

**Plain-English overview of what this notebook does, in order:**

1. Load a sample of the data (same approach as the EDA notebook).
2. Create the **target**: a single column that is `0` for benign and `1` for malicious.
3. Keep only the features we justified in the EDA, drop the rest.
4. Clean missing values (Zeek writes `-` when a value is missing).
5. Turn the `history` text into four simple yes/no flags.
6. **Split first** into train/test, *then* prepare the numbers (this avoids cheating - see note in Step 6).
7. Scale the numbers and convert text columns into numeric columns (one-hot).
8. Save everything so the modelling notebook can just load it.

The output of this notebook is the input to the modelling notebook (`03_models`).

## Setup: imports and folder paths

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import joblib   # used to save Python objects (our prepared data) to disk

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Where the raw dataset lives
DATA_ROOT = r"C:\Users\Asus\OneDrive\Desktop\MSc Cybersecurity -NTU\Major Project\Dataset\iot_23_datasets_small"

# Anchor outputs to the project root so saving works from any working directory
PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "outputs", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)
print("Processed data will be saved to:", PROCESSED_DIR)

Processed data will be saved to: C:\Users\Asus\Desktop\Desktop\MSc Cybersecurity -NTU\Major Project\iot_anomaly_detection\outputs\processed


## Step 1 - Load a sample of the data

This is the **same loading logic as the EDA notebook**: we take up to 50,000 rows from every capture file so that both benign traffic and every attack type are represented. Loading all 325 million rows is unnecessary and would not fit in memory; a representative sample is enough to train and fairly test the models.

In [2]:
ROWS_PER_FILE = 50000   # cap per file -> manageable sample with both classes

all_files = glob.glob(DATA_ROOT + "/**/*.labeled", recursive=True)
print(f"Found {len(all_files)} labeled files")

all_chunks = []
for filepath in all_files:
    # Read column names from the Zeek "#fields" header line
    col_names = []
    with open(filepath) as f:
        for line in f:
            if line.startswith("#fields"):
                col_names = line.strip().split("\t")[1:]
                break

    rows_from_this_file = 0
    for chunk in pd.read_csv(filepath, sep="\t", comment="#",
                              header=None, low_memory=False, chunksize=50000):
        chunk.columns = col_names

        # The last Zeek column merges tunnel_parents + label + detailed_label
        last_col = chunk.columns[-1]
        split = chunk[last_col].str.strip().str.split(r"\s{2,}", expand=True, n=2, regex=True)
        if split.shape[1] == 3:
            split.columns = ["tunnel_parents", "label", "detailed_label"]
            chunk = chunk.drop(columns=[last_col])
            chunk = pd.concat([chunk, split], axis=1)

        chunk["label"] = chunk["label"].str.lower().str.strip()
        all_chunks.append(chunk)
        rows_from_this_file += len(chunk)
        if rows_from_this_file >= ROWS_PER_FILE:
            break

df = pd.concat(all_chunks, ignore_index=True)
print(f"Total rows loaded: {len(df):,}")
print(df["label"].value_counts())

Found 23 labeled files
Total rows loaded: 746,662
label
malicious    637485
benign       109177
Name: count, dtype: int64


## Step 2 - Create the target (what the model predicts)

A supervised model learns to predict one column - the **target**. We are doing **binary** detection, so the target is simply:

- `0` = benign (normal traffic)
- `1` = malicious (an attack)

We build it from the `label` column. The detailed attack name (`detailed_label`) is kept aside only for later analysis and mitigation discussion - the model itself just predicts 0 or 1.

In [3]:
# 1 if malicious, 0 if benign
y = (df["label"] == "malicious").astype(int)

print("Target distribution (0 = benign, 1 = malicious):")
print(y.value_counts())
print(f"\nMalicious proportion: {y.mean():.1%}")
print("\nIf one class is much larger than the other, that is 'class imbalance'.")
print("We handle it later with class_weight in the models, and by using")
print("PR-AUC / F1 instead of plain accuracy when we evaluate.")

Target distribution (0 = benign, 1 = malicious):
label
1    637485
0    109177
Name: count, dtype: int64

Malicious proportion: 85.4%

If one class is much larger than the other, that is 'class imbalance'.
We handle it later with class_weight in the models, and by using
PR-AUC / F1 instead of plain accuracy when we evaluate.


## Step 3 - Keep the useful columns, drop the rest

Not every column helps. Some are **identifiers** (like IP addresses) that would let the model memorise *which capture* a row came from instead of learning *attack behaviour* - that is called **data leakage** and it makes results look better than they really are. We drop those.

**We keep** (justified in the EDA):

- Categorical (text) features: `proto`, `service`, `conn_state`
- Numeric features: `duration`, `orig_bytes`, `resp_bytes`, `missed_bytes`, `orig_pkts`, `resp_pkts`, `orig_ip_bytes`, `resp_ip_bytes`, `id.resp_p` (destination port)
- `history` - which we turn into flags in Step 5

**We drop:** `ts` (timestamp - leakage), `uid`, `id.orig_h`/`id.resp_h` (IP addresses), `id.orig_p` (source port - random), `local_orig`, `local_resp`, `tunnel_parents` (almost always empty), and the label columns (which became our target).

In [4]:
categorical_features = ["proto", "service", "conn_state"]
numeric_features = ["duration", "orig_bytes", "resp_bytes", "missed_bytes",
                    "orig_pkts", "resp_pkts", "orig_ip_bytes", "resp_ip_bytes",
                    "id.resp_p"]

# Keep a copy of the columns we will work with (history handled in Step 5)
X = df[categorical_features + numeric_features + ["history"]].copy()
print("Working feature table shape:", X.shape)
X.head(3)

Working feature table shape: (746662, 13)


,proto,service,conn_state,duration,orig_bytes,resp_bytes,missed_bytes,orig_pkts,resp_pkts,orig_ip_bytes,resp_ip_bytes,id.resp_p,history
0,udp,-,SF,0.114184,48,48,0,1,1,76,76,123,Dd
1,udp,-,S0,160.367579,7536,0,0,24,0,8208,0,1900,D
2,udp,-,SF,0.016986,48,48,0,1,1,76,76,123,Dd


## Step 4 - Clean missing values

Zeek writes the text `-` when a value is missing. Computers can't do maths on `-`, so:

- For **numeric** columns: turn `-` (and anything non-numeric) into a real number. Missing becomes `0` (e.g. 0 bytes sent).
- For **text** columns: replace `-` with the word `unknown` so it becomes a normal category.

We do this cleaning the same way for every row, so it is safe to do before the split.

In [5]:
# Numeric columns: convert to numbers; "-" and bad values become NaN, then fill with 0
for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors="coerce").fillna(0)

# Text columns: replace "-" with "unknown"
for col in categorical_features:
    X[col] = X[col].replace("-", "unknown").fillna("unknown")

print("Any missing values left?", X[categorical_features + numeric_features].isna().any().any())
X[categorical_features + numeric_features].dtypes

Any missing values left? False


proto             object
service           object
conn_state        object
duration         float64
orig_bytes       float64
resp_bytes       float64
missed_bytes       int64
orig_pkts          int64
resp_pkts          int64
orig_ip_bytes      int64
resp_ip_bytes      int64
id.resp_p          int64
dtype: object

## Step 5 - Turn `history` into four simple flags

`history` is a short code describing the TCP handshake, e.g. `ShADadFf`. Each letter is an event. Instead of feeding the model messy text, we extract four yes/no (1/0) flags that matter for spotting attacks:

- `has_syn` - a connection was **attempted** (S). Lots of SYNs with nothing else = scanning.
- `has_ack` - the connection was **acknowledged** (A).
- `has_fin` - the connection **closed cleanly** (F).
- `has_rst` - the connection was **reset / refused** (R). Common in failed/blocked attempts.

We check letters case-insensitively (Zeek uses upper case for the originator and lower case for the responder; for our purposes 'did this event happen at all' is enough).

In [6]:
hist = X["history"].fillna("").str.lower()

X["has_syn"] = hist.str.contains("s").astype(int)
X["has_ack"] = hist.str.contains("a").astype(int)
X["has_fin"] = hist.str.contains("f").astype(int)
X["has_rst"] = hist.str.contains("r").astype(int)

flag_features = ["has_syn", "has_ack", "has_fin", "has_rst"]

# We no longer need the raw history text
X = X.drop(columns=["history"])

print(X[flag_features].sum().rename("count of 1s per flag"))
X[flag_features].head(3)

has_syn    645811
has_ack      5318
has_fin      5309
has_rst      1998
Name: count of 1s per flag, dtype: int64


,has_syn,has_ack,has_fin,has_rst
0,0,0,0,0
1,0,0,0,0
2,0,0,0,0


## Step 6 - Split into training and test sets (do this FIRST)

We hold back 20% of the data as a **test set** - the closed exam we use at the very end to measure real performance. The other 80% is the **training set** the model learns from.

**Why split before scaling/encoding (Step 7)?** If we prepared the numbers using the whole dataset, information from the test rows would leak into the preparation and the model would effectively get a peek at the exam. So we split now, learn all preparation settings from the training rows only, then apply them to the test rows.

`stratify=y` keeps the same benign/malicious ratio in both sets, which matters when the classes are imbalanced. `random_state=42` just makes the split reproducible.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,        # keep the same class balance in train and test
    random_state=42,   # reproducible split
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows:     {len(X_test):,}")
print(f"Malicious % in train: {y_train.mean():.1%}   in test: {y_test.mean():.1%}")

Training rows: 597,329
Test rows:     149,333
Malicious % in train: 85.4%   in test: 85.4%


## Step 7 - Prepare the numbers (fit on train, apply to test)

Two jobs, done together by a `ColumnTransformer` (a tidy way to apply different preparation to different columns):

1. **Scale numeric columns** with `RobustScaler`. Scaling puts features on a comparable range so big numbers (like byte counts) don't dominate small ones. *Robust* scaling uses the median and is not thrown off by the extreme values attacks produce.
2. **One-hot encode text columns**. A model needs numbers, not words. One-hot turns `proto = tcp/udp/icmp` into separate 0/1 columns. `min_frequency=0.01` groups rare categories together (important for `service`, which has many rare values) so we don't create hundreds of columns. `handle_unknown='ignore'` means a category only seen in the test set won't crash anything.

The four history flags are already 0/1, so we pass them through untouched.

Crucially we call `fit_transform` on **train** (learn the settings AND apply them) but only `transform` on **test** (apply the same settings, learn nothing new).

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore",
                              min_frequency=0.01,
                              sparse_output=False), categorical_features),
        ("flags", "passthrough", flag_features),
    ],
    remainder="drop",
)

# Learn settings from TRAIN, then apply to both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)   # transform only - no peeking

feature_names = preprocessor.get_feature_names_out()

print("Processed training data shape:", X_train_processed.shape)
print("Processed test data shape:    ", X_test_processed.shape)
print(f"\nNumber of model input columns after encoding: {len(feature_names)}")
print("First few column names:", list(feature_names[:8]))

Processed training data shape: (597329, 22)
Processed test data shape:     (149333, 22)

Number of model input columns after encoding: 22
First few column names: ['num__duration', 'num__orig_bytes', 'num__resp_bytes', 'num__missed_bytes', 'num__orig_pkts', 'num__resp_pkts', 'num__orig_ip_bytes', 'num__resp_ip_bytes']


## Step 8 - Save everything for the modelling notebook

We save two things:

1. The prepared data (`X_train_processed`, `X_test_processed`, `y_train`, `y_test`) and the column names - so `03_models` can load them directly.
2. The fitted `preprocessor` - so the exact same preparation can be reused later (e.g. on brand-new live traffic) without re-learning settings.

`joblib` is the standard way to save scikit-learn objects to disk.

In [9]:
data_path = os.path.join(PROCESSED_DIR, "train_test_data.joblib")
prep_path = os.path.join(PROCESSED_DIR, "preprocessor.joblib")

joblib.dump({
    "X_train": X_train_processed,
    "X_test": X_test_processed,
    "y_train": y_train.to_numpy(),
    "y_test": y_test.to_numpy(),
    "feature_names": feature_names,
}, data_path)

joblib.dump(preprocessor, prep_path)

print("Saved:")
print(" -", data_path)
print(" -", prep_path)
print("\nNext notebook (03_models) will load train_test_data.joblib and train")
print("Logistic Regression (baseline) and Random Forest (primary, with SHAP).")

Saved:
 - C:\Users\Asus\Desktop\Desktop\MSc Cybersecurity -NTU\Major Project\iot_anomaly_detection\outputs\processed\train_test_data.joblib
 - C:\Users\Asus\Desktop\Desktop\MSc Cybersecurity -NTU\Major Project\iot_anomaly_detection\outputs\processed\preprocessor.joblib

Next notebook (03_models) will load train_test_data.joblib and train
Logistic Regression (baseline) and Random Forest (primary, with SHAP).
